<a href="https://colab.research.google.com/github/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab/blob/main/RVC_For_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVC for Colab

在 Colab 上运行官方 [RVC-Project/Retrieval-based-Voice-Conversion-WebUI](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI)，默认固定在 2.3.260718 之后的 main（2026-08-04，81eed5e），包含 release 说明中的全部更新（如 PyMSS 人声分离）。
Runs the official [RVC-Project/Retrieval-based-Voice-Conversion-WebUI](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI) on Colab, pinned by default to 2.3.260718 + 2026-08-04 main (81eed5e), which includes everything in the release notes (such as PyMSS vocal separation).

1. 选择 GPU 运行时：`代码执行程序 > 更改运行时类型 > T4 GPU`。
   Select a GPU runtime: `Runtime > Change runtime type > T4 GPU`.
2. 按顺序运行“准备环境”中的单元格，然后选择 **WebUI** 或 **命令行训练** 其中一种方式。
   Run the cells under "Setup" in order, then use either the **WebUI** or the **command-line training** section.

项目主页 / Project page: https://github.com/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab

# 准备环境 / Setup

In [ ]:
# @title 克隆 RVC 官方仓库 / Clone the official RVC repository
# @markdown 上游的 tag、分支或完整的提交 SHA，默认值为本笔记本适配的版本 / Upstream tag, branch or full commit SHA; the default is the version this notebook is written for
RVC_REF = "81eed5e8f68b6bed1789f682fe78cdd324495afc"  # @param {type:"string"}

%cd /content
!rm -rf /content/Retrieval-based-Voice-Conversion-WebUI
!git init -q /content/Retrieval-based-Voice-Conversion-WebUI
%cd /content/Retrieval-based-Voice-Conversion-WebUI
!git remote add origin https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git
!git fetch -q --depth=1 origin "{RVC_REF}" && git checkout -q FETCH_HEAD
!git log -1 --format="%h %ad %s" --date=short

In [ ]:
# @title 安装依赖 / Install dependencies
import sys
if sys.version_info[:2] != (3, 12):
    print(f"警告：RVC 2.3 面向 Python 3.12，当前为 {sys.version.split()[0]} / "
          f"Warning: RVC 2.3 targets Python 3.12, this runtime is {sys.version.split()[0]}")

%cd /content/Retrieval-based-Voice-Conversion-WebUI
# 上游要求 torch < 2.8，Colab 自带的版本更新，需要替换 / Upstream needs torch < 2.8, newer than what Colab ships
!pip install -q torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu128
# 去掉 requirements 中的国内镜像，改用 PyPI / Drop the mirror pinned in the requirements file and use PyPI
!grep -v "^--index-url" requirments_cu128_py312.txt > /tmp/requirements.txt
!pip install -q -r /tmp/requirements.txt

In [ ]:
# @title 下载模型 / Download models
# @markdown 下载 v1 底模（约 0.7 GB） / Also download v1 pretrained models (about 0.7 GB)
DOWNLOAD_V1 = False  # @param {type:"boolean"}
# @markdown 下载 PyMSS 人声分离模型（约 2 GB，用于清理数据集或推理前分离人声） / Download PyMSS vocal separation models (about 2 GB, for cleaning datasets or separating vocals before inference)
DOWNLOAD_PYMSS = False  # @param {type:"boolean"}

import os, zipfile
from huggingface_hub import hf_hub_download, snapshot_download

os.chdir("/content/Retrieval-based-Voice-Conversion-WebUI")
HF_REPO = "lj1995/VoiceConversionWebUI"
# WebUI 训练只提供 40k / 48k / The WebUI only trains at 40k and 48k
patterns = ["hubert_base/*", "pretrained_v2/*40k.pth", "pretrained_v2/*48k.pth"]
if DOWNLOAD_V1:
    patterns += ["pretrained/*40k.pth", "pretrained/*48k.pth"]
if DOWNLOAD_PYMSS:
    patterns.append("pymss_weights/*")
snapshot_download(HF_REPO, allow_patterns=patterns, local_dir="assets")
hf_hub_download(HF_REPO, "rmvpe.pt", local_dir="assets/rmvpe")
mute_zip = hf_hub_download(HF_REPO, "mute.zip", local_dir="/content/.model-downloads")
with zipfile.ZipFile(mute_zip) as z:
    z.extractall("logs")
print("Done:", sorted(os.listdir("assets")), sorted(os.listdir("logs")))

In [ ]:
# @title 挂载谷歌云盘 / Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title 从谷歌云盘解压数据集到 /content/dataset / Unzip dataset from Google Drive to /content/dataset
# @markdown 数据集 zip 路径（zip 内直接放音频文件） / Path of the dataset zip (audio files at the top level)
DATASET = "/content/drive/MyDrive/datasets/RVC/src.zip"  # @param {type:"string"}

!mkdir -p /content/dataset
!unzip -o -q -d /content/dataset "{DATASET}"
!ls /content/dataset | head

In [ ]:
# @title 从谷歌云盘恢复模型（首次使用无需执行） / Restore a model from Google Drive (skip on first use)
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 要恢复的 epoch；“仅保存最新”时填 2333333 / Epoch to restore; use 2333333 if "only save latest" was on
MODELEPOCH = 2333333  # @param {type:"integer"}
# @markdown 备份所在目录 / Backup folder on Drive
BACKUP_DIR = "/content/drive/MyDrive/RVC_backup"  # @param {type:"string"}
# @markdown 2025 年旧版笔记本的备份（文件在 MyDrive 根目录，且 G/D 文件名互换） / Backup made by the 2025 notebook (files in MyDrive root, G/D names swapped)
LEGACY_BACKUP = False  # @param {type:"boolean"}

import glob, os, shutil

os.chdir("/content/Retrieval-based-Voice-Conversion-WebUI")
exp_dir = f"logs/{MODELNAME}"
os.makedirs(exp_dir, exist_ok=True)
os.makedirs("assets/weights", exist_ok=True)

if LEGACY_BACKUP:
    src = "/content/drive/MyDrive"
    shutil.copy(f"{src}/{MODELNAME}_D_{MODELEPOCH}.pth", f"{exp_dir}/G_{MODELEPOCH}.pth")
    shutil.copy(f"{src}/{MODELNAME}_G_{MODELEPOCH}.pth", f"{exp_dir}/D_{MODELEPOCH}.pth")
    shutil.copy(f"{src}/{MODELNAME}{MODELEPOCH}.pth", f"assets/weights/{MODELNAME}.pth")
    extra = glob.glob(f"{src}/added_*{MODELNAME}*.index") + glob.glob(f"{src}/total_*.npy")
else:
    src = f"{BACKUP_DIR}/{MODELNAME}"
    for gd in ("G", "D"):
        shutil.copy(f"{src}/{gd}_{MODELEPOCH}.pth", exp_dir)
    for f in glob.glob(f"{src}/{MODELNAME}*.pth"):
        shutil.copy(f, "assets/weights")
    extra = glob.glob(f"{src}/*.index") + glob.glob(f"{src}/*.npy") + glob.glob(f"{src}/config.json")
for f in extra:
    shutil.copy(f, exp_dir)
print("Restored to", exp_dir, sorted(os.listdir(exp_dir)))

# 方式一：WebUI / Option 1: WebUI

In [ ]:
# @title 方案一：Gradio 公网链接 / Plan A: Gradio share link
%cd /content/Retrieval-based-Voice-Conversion-WebUI
!python webui.py --colab

In [ ]:
# @title 方案二：ngrok 隧道 / Plan B: ngrok tunnel
# @markdown 在 https://dashboard.ngrok.com 获取 Authtoken / Get your authtoken at https://dashboard.ngrok.com
Authtoken = ""  # @param {type:"string"}

%cd /content/Retrieval-based-Voice-Conversion-WebUI
!pip -q install pyngrok
from pyngrok import ngrok

if not Authtoken:
    raise ValueError("请先填写 ngrok Authtoken / Please fill in your ngrok authtoken")
ngrok.set_auth_token(Authtoken)
public_url = ngrok.connect(7865, "http")
print("ngrok 公网访问地址 / Public URL:", public_url.public_url)
!python webui.py --port 7865 --noautoopen

# 方式二：命令行训练 / Option 2: command-line training
若运行 WebUI 时提示该代码被禁止执行，可以用下面的单元格直接训练，命令参考官方 [CLI 文档](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI/blob/main/docs/en/cli.md)。
If Colab refuses to run the WebUI, train with the cells below. The commands follow the official [CLI guide](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI/blob/main/docs/en/cli.md).

所有单元格中的模型名、采样率和版本需保持一致。
Keep the model name, sample rate and version the same in every cell.

In [ ]:
# @title 音频预处理 / Preprocess audio
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 采样率 / Sample rate
SAMPLERATE = "40k"  # @param ["40k", "48k"]
# @markdown 使用的进程数 / Number of processes
THREADCOUNT = 8  # @param {type:"integer"}

%cd /content/Retrieval-based-Voice-Conversion-WebUI
SR_HZ = int(SAMPLERATE[:-1]) * 1000
!mkdir -p logs/{MODELNAME}
!python train/preprocess.py /content/dataset {SR_HZ} {THREADCOUNT} logs/{MODELNAME} False 3.7

In [ ]:
# @title 提取音高与特征 / Extract pitch and features
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 版本 / Version
VERSION = "v2"  # @param ["v2", "v1"]
# @markdown 音高提取算法，2.3 训练只支持 rmvpe 和 pm，rmvpe_gpu 最快且效果最好 / Pitch extraction method; 2.3 only trains with rmvpe or pm, and rmvpe_gpu is the fastest and usually the best
ALGO = "rmvpe_gpu"  # @param ["rmvpe_gpu", "rmvpe", "pm"]
# @markdown 使用的进程数（rmvpe_gpu 不使用） / Number of processes (not used by rmvpe_gpu)
THREADCOUNT = 8  # @param {type:"integer"}

%cd /content/Retrieval-based-Voice-Conversion-WebUI
if ALGO == "rmvpe_gpu":
    !python train/dataset/extract_f0.py cuda 1 0 0 logs/{MODELNAME} true
else:
    !python train/dataset/extract_f0.py cpu logs/{MODELNAME} {THREADCOUNT} {ALGO}
!python train/dataset/extract_hubert_feature.py cuda:0 1 0 0 logs/{MODELNAME} {VERSION} true

In [ ]:
# @title 训练模型 / Train the model
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 采样率 / Sample rate
SAMPLERATE = "40k"  # @param ["40k", "48k"]
# @markdown 版本 / Version
VERSION = "v2"  # @param ["v2", "v1"]
# @markdown 批大小 / Batch size
BATCHSIZE = 8  # @param {type:"integer"}
# @markdown 总 epoch 数 / Total epochs
MODELEPOCH = 200  # @param {type:"integer"}
# @markdown 保存间隔（epoch） / Save every N epochs
EPOCHSAVE = 10  # @param {type:"integer"}
# @markdown 是否仅保存最新的 G/D 检查点（文件名为 G_2333333.pth） / Only keep the latest G/D checkpoint (saved as G_2333333.pth)
ONLYLATEST = True  # @param {type:"boolean"}
# @markdown 是否把训练集缓存到显存（数据集较小时可加速） / Cache the dataset in GPU memory (faster for small datasets)
CACHEDATA = False  # @param {type:"boolean"}
# @markdown 每次保存时都导出一个小模型到 assets/weights / Export a small model to assets/weights at every save
SAVEWEIGHTS = True  # @param {type:"boolean"}
# @markdown 在下方显示 TensorBoard / Show TensorBoard below
TENSORBOARD = True  # @param {type:"boolean"}

import json, os, random

os.chdir("/content/Retrieval-based-Voice-Conversion-WebUI")
# 与 webui.py 的 click_train 相同：生成 filelist.txt 和 config.json / Same as click_train in webui.py: write filelist.txt and config.json
root = os.getcwd()
exp_dir = f"{root}/logs/{MODELNAME}"
fea_dim = 256 if VERSION == "v1" else 768
dirs = {"wav": f"{exp_dir}/0_gt_wavs", "fea": f"{exp_dir}/3_feature{fea_dim}",
        "f0": f"{exp_dir}/2a_f0", "f0nsf": f"{exp_dir}/2b-f0nsf"}
names = set.intersection(*(set(n.split(".")[0] for n in os.listdir(d)) for d in dirs.values()))
if not names:
    raise RuntimeError("没有可用于训练的音频，请先完成预处理和特征提取 / No usable samples, run preprocessing and feature extraction first")
lines = [f"{dirs['wav']}/{n}.wav|{dirs['fea']}/{n}.npy|{dirs['f0']}/{n}.wav.npy|{dirs['f0nsf']}/{n}.wav.npy|0" for n in names]
mute = f"{root}/logs/mute"
lines += [f"{mute}/0_gt_wavs/mute{SAMPLERATE}.wav|{mute}/3_feature{fea_dim}/mute.npy|{mute}/2a_f0/mute.wav.npy|{mute}/2b-f0nsf/mute.wav.npy|0"] * 2
random.shuffle(lines)
with open(f"{exp_dir}/filelist.txt", "w", encoding="utf8") as f:
    f.write("\n".join(lines))
config_file = f"v1/{SAMPLERATE}.json" if VERSION == "v1" or SAMPLERATE == "40k" else f"v2/{SAMPLERATE}.json"
if not os.path.exists(f"{exp_dir}/config.json"):
    with open(f"configs/{config_file}", encoding="utf8") as f:
        cfg = json.load(f)
    with open(f"{exp_dir}/config.json", "w", encoding="utf8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=4, sort_keys=True)
        f.write("\n")
print(f"{len(names)} samples, config {config_file}")

if TENSORBOARD:
    %load_ext tensorboard
    %tensorboard --logdir logs
PRETRAINED = "assets/pretrained_v2" if VERSION == "v2" else "assets/pretrained"
!python train/train.py -e {MODELNAME} -sr {SAMPLERATE} -f0 1 -bs {BATCHSIZE} -g 0 -te {MODELEPOCH} -se {EPOCHSAVE} -pg {PRETRAINED}/f0G{SAMPLERATE}.pth -pd {PRETRAINED}/f0D{SAMPLERATE}.pth -l {int(ONLYLATEST)} -c {int(CACHEDATA)} -sw {int(SAVEWEIGHTS)} -v {VERSION}

In [ ]:
# @title 训练特征索引 / Train the feature index
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 版本 / Version
VERSION = "v2"  # @param ["v2", "v1"]

import os
%cd /content/Retrieval-based-Voice-Conversion-WebUI
!python train/train_index.py {MODELNAME} {VERSION} assets/indices {os.cpu_count()}

# 备份 / Backup

In [ ]:
# @title 备份模型到谷歌云盘 / Back up the model to Google Drive
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 要备份的 epoch；“仅保存最新”时填 2333333 / Epoch to back up; use 2333333 if "only save latest" was on
MODELEPOCH = 2333333  # @param {type:"integer"}
# @markdown 备份目录 / Backup folder on Drive
BACKUP_DIR = "/content/drive/MyDrive/RVC_backup"  # @param {type:"string"}

import glob, os, shutil

os.chdir("/content/Retrieval-based-Voice-Conversion-WebUI")
exp_dir = f"logs/{MODELNAME}"
dst = f"{BACKUP_DIR}/{MODELNAME}"
os.makedirs(dst, exist_ok=True)
files = [f"{exp_dir}/G_{MODELEPOCH}.pth", f"{exp_dir}/D_{MODELEPOCH}.pth", f"{exp_dir}/config.json"]
files += glob.glob(f"{exp_dir}/added_*.index") + glob.glob(f"{exp_dir}/total_*.npy")
files += glob.glob(f"assets/weights/{MODELNAME}*.pth")
for f in files:
    if os.path.exists(f):
        shutil.copy(f, dst)
        print("copied", f)
    else:
        print("missing", f)
print("Backup folder:", dst)